In [ ]:
print(df.isnull().sum())

NameError: name 'df' is not defined

In [ ]:
print(df.fillna(df['director']=='unknown'))

In [ ]:
print(df.isnull().sum())

In [ ]:
def remove_nulls(df, cols):
    return df.dropna(subset = cols)

df = remove_nulls(df, ["director", "cast", "country", "date_added", "rating", "duration", "parsed_date"])

In [ ]:
print(df.isnull().sum())

In [ ]:
df['director'].head

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [ ]:
print(df['director'])

In [ ]:
df['listed_in'].apply(lambda x:[i.replace(" ","") for i in x.split(",")])

In [ ]:
print(df['director'].iloc[0])
print(type(df['director'].iloc[0]))

In [ ]:
df['director'] = df['director'].apply(lambda x: [i.replace(" ", "") for i in x] if isinstance(x, list) else [])



In [ ]:
print(df['director'].head(5))

In [ ]:
df.head(25)

In [ ]:
import pandas as pd

# Step 2: Split cast into list
def preprocess_cast(df, col='cast'):
    df[col] = df[col].apply(lambda x: [i.strip() for i in x.split(",")] if isinstance(x, str) else [])
    return df

# Step 3: Split country into list
def preprocess_country(df, col='country'):
    df[col] = df[col].apply(lambda x: [i.strip() for i in x.split(",")] if isinstance(x, str) else [])
    return df

# Step 4: Extract year, month, day_of_week from parsed_date
def add_date_features(df, col='parsed_date'):
    df['year_added'] = df[col].dt.year
    df['month_added'] = df[col].dt.month
    df['day_of_week'] = df[col].dt.day_name()
    return df

def preprocess_description(df, col='description', out_col='description_tokens'):
    df[out_col] = df[col].apply(
        lambda x: x.split() if isinstance(x, str) else []
    )
    return df

# Step 5: Clean duration into numeric columns
def preprocess_duration(df, col='duration'):
    def convert_duration(val):
        if isinstance(val, str):
            if "min" in val:
                return int(val.replace("min", "").strip()), None
            elif "Season" in val:
                return None, int(val.replace("Seasons", "").replace("Season", "").strip())
        return None, None
    
    df['duration_minutes'], df['num_seasons'] = zip(*df[col].map(convert_duration))
    return df

# Step 6: Split listed_in (genres) into list
def preprocess_genres(df, col='listed_in'):
    df['genres'] = df[col].apply(lambda x: [i.strip() for i in x.split(",")] if isinstance(x, str) else [])
    return df


In [ ]:
# Assuming df is your Netflix dataframe
df = preprocess_cast(df)
df = preprocess_country(df)
df = add_date_features(df)
df = preprocess_duration(df)
df = preprocess_description(df)
df = preprocess_genres(df)


In [ ]:
df.head(20)

In [ ]:
# Every row should now have proper list types:
assert df['cast'].map(lambda x: isinstance(x, list)).all()
assert df['country'].map(lambda x: isinstance(x, list)).all()
assert df['genres'].map(lambda x: isinstance(x, list)).all()
assert df['description_tokens'].map(lambda x: isinstance(x, list)).all()

# Duration columns should be nullable integers (or int if fillna_zero=True)
print(df['duration_minutes'].dtype, df['num_seasons'].dtype)

# Peek a few rows
print(df[['type','title','duration','duration_minutes','num_seasons','genres']].head())



In [ ]:
#Concatenation of multiple columns

df['tags'] =  df['description_tokens'] + df['genres'] + df['cast'] + df['director']

In [ ]:
new_df = df[['show_id', 'title', 'tags']]

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

In [ ]:
new_df.head(10)


In [ ]:
new_df['tags'][0]

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['listed_in'].str.split(', '))
genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_)

In [ ]:
df_final = pd.concat([df, genre_df], axis=1)

In [ ]:
df_final.columns

In [ ]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nMissing values:\n", df.isna().sum().sort_values(ascending=False))
print("\nDuplicates:", df.duplicated(subset=['title','type','release_year']).sum())


In [ ]:
for col in ['title', 'director', 'cast', 'country', 'listed_in', 'rating', 'description']:
    if col in df_final.columns:
        df_final[col] = df_final[col].astype(str).str.strip().replace({'nan':'', 'None':''})

In [ ]:
## Find similarity between movies

from sklearn.metrics.pairwise import cosine_similarity
cos_sim_matrix = cosine_similarity(genre_df.values)

In [ ]:
sim_df = pd.DataFrame(cos_sim_matrix, index=df['title'], columns=df['title'])
print(sim_df.tail(10))

In [ ]:
print(sim_df.index[:20]) 

In [ ]:
def recommend_movie(movie_title, sim_matrix, top_n=5):
    if movie_title not in sim_matrix.columns:
        return f"Movie '{movie_title}' not found in dataset!"
    
    similar_movies = sim_matrix[movie_title].sort_values(ascending=False)
    return similar_movies.iloc[1:top_n+1]
    
print(recommend_movie("The Darkest Hour", sim_df, top_n=3))

In [ ]:
print(recommend_movie("2012", sim_df, top_n=10))


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

def build_similarity_matrix(df):
    tfidf = TfidfVectorizer(stop_words="english")
    tfidf_matrix = tfidf.fit_transform(df['features'])
    cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
    return cosine_sim

In [3]:
import pandas as pd

def recommend_movie(title, df, cosine_sim, top_n=10):
    title = title.lower().strip()
    indices = pd.Series(df.index, index=df['title']).drop_duplicates()
    
    if title not in indices:
        return f"'{title}' not found in dataset."
    
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]  # skip itself
    
    movie_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[movie_indices]

In [11]:
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('.')))

In [13]:
def recommend_movie(movie_name, sim_df, top_n=10):
    movie_name = movie_name.lower().strip()
    if movie_name not in sim_df.index:
        return "Movie not found in dataset."

    # Sort similar movies
    similar_scores = sim_df[movie_name].sort_values(ascending=False)
    
    # Exclude the movie itself
    recommendations = similar_scores.iloc[1:top_n+1]
    return recommendations

# Example
print(recommend_movie("the lord of the rings: the two towers", sim_df, 5))

NameError: name 'sim_df' is not defined